# Embedding 11 video bộ cũ còn thiếu (Google Colab)

Bộ cũ (L21–L30) có 873 video nhưng chỉ 862 video có vector trong chỉ mục; 11 video dưới đây **chưa có vector** nên không tìm được.
Notebook làm trọn cả chuỗi cho 11 video đó (tổng ~800 MB, ~73 phút video, khoảng 1.300 keyframe):

**lấy đúng 11 file từ `videos.zip` trên HF (đọc từng đoạn, không tải cả 83 GB) → tách keyframe (đúng thuật toán bộ cũ) → embedding PE-Core → upload lên HF**

**Trước khi chạy:** `Runtime → Change runtime type → A100 GPU` (T4 cũng chạy được, chậm hơn).

| Cell | Việc |
|---|---|
| 1 | Cài thư viện + kiểm tra GPU |
| 2 | Đăng nhập Hugging Face (token có quyền **Write**) |
| 3 | Cấu hình (danh sách 11 video) |
| 4 | Nạp CLIP (tách keyframe) + PE-Core (embedding), ~2–3 phút lần đầu |
| 5 | Chạy — ước tính vài phút |

**Kết quả** trên `5clnl/idk`:
- `embeddings_pecore_old_missing/<video>.npz` — `frame_idx` (int32), `pe_core` [n,1280] float32 đã chuẩn hoá, `fps`
- `keyframes_old_missing/<video>.zip` — ảnh keyframe `<video>_frame_<số frame>.jpg` (để hiện ảnh xem trước)

Video nào đã có kết quả trên HF sẽ được bỏ qua, nên chạy lại cell 5 bất cứ lúc nào để làm nốt phần lỗi.


In [ ]:
#@title 1. Cài thư viện & kiểm tra GPU
# Ghim đúng phiên bản đang dùng ở máy local để kết quả khớp với phần đã làm.
!pip install -q "open_clip_torch==3.3.0" "timm==1.0.28" av

import torch
if not torch.cuda.is_available():
    print("CẢNH BÁO: không có GPU -> sẽ chạy rất chậm. Runtime > Change runtime type > chọn GPU.")
else:
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name}, {p.total_memory/1e9:.0f} GB VRAM")


In [ ]:
#@title 2. Đăng nhập Hugging Face
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
from huggingface_hub import login, whoami
login()
print("Đã đăng nhập với tài khoản:", whoami()["name"])


In [ ]:
#@title 3. Cấu hình
REPO = "5clnl/idk"
ZIP_NAME = "videos.zip"                      # file zip video bộ cũ ở gốc repo (873 video)
EMB_DIR = "embeddings_pecore_old_missing"    # kết quả embedding
KF_DIR = "keyframes_old_missing"             # ảnh keyframe (xem trước)
VIDEOS = ["L21_V001", "L26_V074", "L26_V079", "L26_V328", "L26_V340", "L26_V341", "L26_V342", "L26_V353", "L26_V355", "L26_V358", "L26_V371"]
print(f"{len(VIDEOS)} video: {', '.join(VIDEOS)}")


In [ ]:
#@title 4. Nạp 2 model + thuật toán tách keyframe (giống hệt pipeline local) + hàm embedding PE-Core
#@markdown Keyframe: CLIP ViT-L-14 (dfn2b) fp16 · ngưỡng 0.6 · cách nhau tối thiểu 5 frame · lấy mẫu mỗi 5 frame · so với keyframe gần nhất đã chọn.
#@markdown Embedding: PE-Core-bigG-14-448 fp16, chuẩn hoá L2 (giống hệt cách embed 614 video mới).
import itertools, os, queue, shutil, threading, time, zipfile
from collections import deque
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import cv2, numpy as np, torch, open_clip
from PIL import Image
try:
    import av
except Exception:
    av = None

THRESHOLD, MIN_DELTA, SAMPLE_RATE = 0.6, 5, 5
BATCH_SIZE = 128
QUEUE_MAX = 256
AV_THREADS = 0
DECODER = "auto"                       # thử OpenCV trước, không được thì PyAV
EMB_DIM = 1280
EMBED_BATCH = 64
NUM_LOADERS = max(2, min(8, os.cpu_count() or 2))
LOAD_TIMEOUT = 300

device = "cuda" if torch.cuda.is_available() else "cpu"
pe_dtype = torch.float16 if device == "cuda" else torch.float32

print(f"Đang nạp CLIP ViT-L-14 (dfn2b) lên {device}...")
clip_model, _, clip_pre = open_clip.create_model_and_transforms(model_name="ViT-L-14", pretrained="dfn2b", device="cpu")
if device == "cuda":
    clip_model = clip_model.half()
clip_model = clip_model.to(device).eval()

print("Đang nạp PE-Core-bigG-14-448 (lần đầu tải ~5GB)...")
t0 = time.time()
pe_model, _, pe_pre = open_clip.create_model_and_transforms("hf-hub:timm/PE-Core-bigG-14-448", device="cpu")
if device == "cuda":
    pe_model = pe_model.half()
pe_model = pe_model.to(device).eval()
print(f"Hai model sẵn sàng (PE-Core sau {time.time() - t0:.0f}s).")


@torch.inference_mode()
def encode(frames_bgr):
    tensors = [clip_pre(Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))) for f in frames_bgr]
    batch = torch.stack(tensors).to(device)
    if device == "cuda":
        batch = batch.half()
    feats = clip_model.encode_image(batch)
    feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats.float().cpu().numpy().astype(np.float32)


def _put(q, item, stop):
    while not stop.is_set():
        try:
            q.put(item, timeout=1)
            return True
        except queue.Full:
            continue
    return False


def _frames_cv2(path):
    cap = cv2.VideoCapture(path)
    try:
        idx = 0
        while True:
            if idx % SAMPLE_RATE == 0:
                ok, frame = cap.read()
                if not ok:
                    return
                yield idx, frame
            else:
                if not cap.grab():
                    return
            idx += 1
    finally:
        cap.release()


def _frames_av(path):
    if av is None:
        raise RuntimeError("chưa cài PyAV (pip install av)")
    with av.open(path) as container:
        stream = container.streams.video[0]
        stream.thread_type = "AUTO"
        if AV_THREADS:
            stream.codec_context.thread_count = AV_THREADS
        for idx, fr in enumerate(container.decode(stream)):
            if idx % SAMPLE_RATE == 0:
                yield idx, fr.reformat(format="bgr24", src_colorspace="ITU601").to_ndarray()


def _probe(path):
    total, fps = 0, 0.0
    cap = cv2.VideoCapture(path)
    if cap.isOpened():
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = cap.get(cv2.CAP_PROP_FPS) or 0.0
    cap.release()
    if total <= 0 and av is not None:
        try:
            with av.open(path) as c:
                st = c.streams.video[0]
                fps = float(st.average_rate or 0) or fps
                total = int(st.frames or (float(st.duration * st.time_base) * fps if st.duration else 0))
        except Exception:
            pass
    return total, (fps or 25.0)


def _reader(path, q, stop, info):
    order = {"auto": ["cv2", "av"], "cv2": ["cv2"], "av": ["av"]}[DECODER]
    try:
        for name in order:
            gen = _frames_cv2(path) if name == "cv2" else _frames_av(path)
            try:
                first = next(gen)
            except StopIteration:
                info["log"].append(f"{name}: không đọc được frame nào")
                continue
            except Exception as e:
                info["log"].append(f"{name}: lỗi {type(e).__name__}: {e}")
                continue
            info["backend"] = name
            try:
                if _put(q, first, stop):
                    for item in gen:
                        if not _put(q, item, stop):
                            break
            except Exception as e:
                info["log"].append(f"{name}: dừng giữa chừng: {type(e).__name__}: {e}")
            finally:
                gen.close()
            break
    finally:
        _put(q, None, stop)


def extract_keyframes(video_path, out_dir):
    name = os.path.splitext(os.path.basename(video_path))[0]
    total, src_fps = _probe(video_path)
    info = {"backend": None, "log": []}
    q = queue.Queue(maxsize=QUEUE_MAX)
    stop = threading.Event()
    threading.Thread(target=_reader, args=(video_path, q, stop, info), daemon=True).start()
    state = {"prev": None, "last_kf": -MIN_DELTA, "saved": 0, "sampled": 0, "last_idx": 0}

    def flush(indices, frames):
        feats = encode(frames)
        for k, idx in enumerate(indices):
            f = feats[k]
            if state["prev"] is None:
                keep = True
            else:
                pn = float(np.linalg.norm(state["prev"]))
                diff = 0.0 if pn < 1e-8 else float(np.linalg.norm(f - state["prev"])) / pn
                keep = diff > THRESHOLD and (idx - state["last_kf"]) >= MIN_DELTA
            if keep:
                cv2.imwrite(os.path.join(out_dir, f"{name}_frame_{idx:06d}.jpg"), frames[k])
                state["prev"], state["last_kf"] = f, idx
                state["saved"] += 1

    t0 = time.time()
    b_idx, b_frames = [], []
    try:
        while True:
            item = q.get()
            if item is None:
                break
            idx, frame = item
            b_idx.append(idx)
            b_frames.append(frame)
            state["last_idx"] = idx
            state["sampled"] += 1
            if len(b_frames) >= BATCH_SIZE:
                flush(b_idx, b_frames)
                b_idx, b_frames = [], []
        if b_frames:
            flush(b_idx, b_frames)
    finally:
        stop.set()
    if state["saved"] == 0:
        raise RuntimeError("không đọc được frame nào (" + " | ".join(info["log"]) + ")")
    return {"video": name, "decoder": info["backend"], "keyframes": state["saved"], "frames_read": state["last_idx"] + 1,
            "frames_declared": total, "source_fps": src_fps, "seconds": round(time.time() - t0, 1)}


def _load_batch(batch):
    return torch.stack([pe_pre(Image.open(p).convert("RGB")) for p in batch])


@torch.inference_mode()
def embed_paths(paths):
    """Embedding PE-Core (chuẩn hoá L2) cho danh sách ảnh, giữ nguyên thứ tự. Đọc ảnh bằng THREAD, có timeout."""
    out = np.empty((len(paths), EMB_DIM), np.float32)
    batches = [paths[i:i + EMBED_BATCH] for i in range(0, len(paths), EMBED_BATCH)]
    it = iter(batches)
    ex = ThreadPoolExecutor(max_workers=max(1, NUM_LOADERS))
    k = 0
    try:
        window = deque(ex.submit(_load_batch, b) for b in itertools.islice(it, max(2, NUM_LOADERS + 2)))
        while window:
            xb = window.popleft().result(timeout=LOAD_TIMEOUT)
            nxt = next(it, None)
            if nxt is not None:
                window.append(ex.submit(_load_batch, nxt))
            f = pe_model.encode_image(xb.to(device).to(pe_dtype))
            f = f / f.norm(dim=-1, keepdim=True).clamp_min(1e-8)
            assert f.shape[1] == EMB_DIM, f.shape
            out[k:k + len(f)] = f.float().cpu().numpy()
            k += len(f)
    finally:
        ex.shutdown(wait=False, cancel_futures=True)
    assert k == len(paths), (k, len(paths))
    return out


In [ ]:
#@title 5. Chạy: lấy video từ videos.zip → tách keyframe → embedding → upload
import json, re, shutil, time, zipfile
from pathlib import Path
from huggingface_hub import HfApi, HfFileSystem

api = HfApi()
WORK = Path("/content/old_missing")
WORK.mkdir(parents=True, exist_ok=True)
FRAME_RE = re.compile(r"^(.+?)_frame_(\d+)\.jpg$")


def log(msg):
    print(msg, flush=True)


def open_zip():
    """Mở videos.zip trên HF như một file có thể tua (đọc từng đoạn qua HTTP Range) — KHÔNG tải cả 83 GB."""
    f = HfFileSystem().open(f"datasets/{REPO}/{ZIP_NAME}", "rb", block_size=16 << 20)
    return zipfile.ZipFile(f)


def fetch_video(holder, vid, dst):
    names = [f"videos/{vid}.mp4", f"{vid}.mp4"]
    last = None
    for attempt in range(1, 5):
        try:
            zf = holder[0]
            info = None
            for n in names:
                try:
                    info = zf.getinfo(n)
                    break
                except KeyError:
                    continue
            if info is None:
                raise FileNotFoundError(f"không có {vid}.mp4 trong {ZIP_NAME}")
            tmp = dst.with_name(dst.name + ".part")
            with zf.open(info) as src, open(tmp, "wb") as out:       # zipfile tự kiểm CRC32 khi đọc hết
                shutil.copyfileobj(src, out, 4 << 20)
            if tmp.stat().st_size != info.file_size:
                raise IOError(f"kích thước {tmp.stat().st_size} khác {info.file_size}")
            tmp.replace(dst)
            return info.file_size
        except FileNotFoundError:
            raise
        except Exception as e:
            last = e
            log(f"[{vid}] lấy video lỗi lần {attempt}/4: {type(e).__name__}: {str(e)[:120]}")
            time.sleep(5 * attempt)
            try:
                holder[0] = open_zip()
            except Exception:
                pass
    raise RuntimeError(f"không lấy được {vid}: {last}")


def upload_verified(local_path, path_in_repo, tries=4):
    size = Path(local_path).stat().st_size
    for t in range(1, tries + 1):
        try:
            api.upload_file(path_or_fileobj=str(local_path), path_in_repo=path_in_repo, repo_id=REPO, repo_type="dataset",
                            commit_message=f"{Path(path_in_repo).stem} (Colab, bộ cũ còn thiếu)")
            info = api.get_paths_info(REPO, [path_in_repo], repo_type="dataset")
            if info and info[0].size == size:
                return True
            log(f"   upload {path_in_repo}: kích thước không khớp (lần {t}/{tries})")
        except Exception as e:
            log(f"   upload {path_in_repo} lỗi (lần {t}/{tries}): {str(e)[:150]}")
        time.sleep(10 * t)
    return False


remote = set(api.list_repo_files(REPO, repo_type="dataset"))
todo = [v for v in VIDEOS if f"{EMB_DIR}/{v}.npz" not in remote]
for v in VIDEOS:
    if v not in todo:
        log(f"[bỏ qua] {v}: đã có kết quả trên Hugging Face")
log(f"Cần làm {len(todo)}/{len(VIDEOS)} video")

holder = [None]
if todo:
    holder[0] = open_zip()
    log(f"Đã mở {ZIP_NAME} ({len(holder[0].namelist())} mục) — chỉ đọc đúng phần cần")

done, failed = [], []
t_all = time.time()
for i, vid in enumerate(todo, 1):
    t0 = time.time()
    kf_dir = WORK / "kf" / vid
    try:
        if kf_dir.exists():
            shutil.rmtree(kf_dir)
        kf_dir.mkdir(parents=True)
        mp4 = WORK / f"{vid}.mp4"
        size = fetch_video(holder, vid, mp4)
        log(f"[{i}/{len(todo)}] {vid}: đã lấy video ({size/1e6:.0f} MB, {time.time()-t0:.0f}s)")

        stats = extract_keyframes(str(mp4), str(kf_dir))
        items = sorted((int(m.group(2)), str(p)) for p in kf_dir.glob("*.jpg") if (m := FRAME_RE.match(p.name)) and m.group(1) == vid)
        if len(items) != stats["keyframes"] or not items:
            raise RuntimeError(f"số ảnh trên đĩa {len(items)} khác số keyframe đã lưu {stats['keyframes']}")
        log(f"    {stats['keyframes']} keyframe / {stats['frames_read']:,} frame ({stats['seconds']:.0f}s, {stats['decoder']}, {stats['source_fps']:.2f} fps)")

        embs = embed_paths([p for _, p in items])
        if not np.isfinite(embs).all() or abs(np.linalg.norm(embs, axis=1) - 1).max() > 1e-2:
            raise RuntimeError("embedding không hợp lệ (NaN hoặc chưa chuẩn hoá)")

        out_dir = WORK / "out"
        out_dir.mkdir(exist_ok=True)
        npz = out_dir / f"{vid}.npz"
        tmp_npz = out_dir / f"{vid}.tmp.npz"
        np.savez(tmp_npz, frame_idx=np.array([f for f, _ in items], np.int32), pe_core=embs, fps=np.float32(stats["source_fps"]))
        tmp_npz.replace(npz)
        zp = out_dir / f"{vid}.zip"
        with zipfile.ZipFile(zp, "w", zipfile.ZIP_STORED) as z:
            for _, p in items:
                z.write(p, arcname=Path(p).name)

        # keyframe (ảnh xem trước) lên trước, .npz lên sau: có .npz trên HF = video này đã xong hoàn toàn
        if upload_verified(zp, f"{KF_DIR}/{vid}.zip") and upload_verified(npz, f"{EMB_DIR}/{vid}.npz"):
            done.append((vid, len(items)))
            log(f"    đã upload {len(items)} vector + keyframe ({time.time()-t0:.0f}s)")
        else:
            failed.append(vid)
            log(f"    !! upload lỗi — file còn trong {out_dir}")
    except Exception as e:
        failed.append(vid)
        log(f"[{vid}] !! LỖI: {type(e).__name__}: {e}")
    finally:
        shutil.rmtree(kf_dir, ignore_errors=True)
        (WORK / f"{vid}.mp4").unlink(missing_ok=True)

log("\n================ TỔNG KẾT ================")
for v, n in done:
    log(f"  OK  {v}: {n} vector")
if failed:
    log("  Lỗi (chạy lại cell này để thử lại): " + ", ".join(failed))
else:
    log(f"Tất cả đã xong ({sum(n for _, n in done)} vector, {(time.time()-t_all)/60:.1f} phút). Báo lại để tôi tải về và nạp vào chỉ mục.")
